# 🚀 DGA Open Data Visualization Starter Kit (DGA Group 6)
### 💡 แนวทางและไอเดียการพัฒนาโครงการจากชุดข้อมูลเปิดภาครัฐ (GD Catalog)

สมุดโน้ตเล่มนี้จัดทำขึ้นเพื่อเป็น **Starter Kit** หรือไอเดียเริ่มต้นสำหรับกลุ่มที่ยังนึกไม่ออกว่าจะทำอะไรในระยะเวลาอันสั้น โดยจะพาไปดูตัวอย่างการเชื่อมโยงข้อมูล การทำความสะอาดข้อมูล (Data Cleaning) และการสร้างแผนภาพแบบโต้ตอบได้ (Interactive Visualization) จากหน่วยงานภาครัฐ 2 แห่ง ได้แก่:
1. **กรมธุรกิจพลังงาน** - ข้อมูลการใช้น้ำมันเชื้อเพลิงในภาคขนส่งรายจังหวัด (ปี 2561 - 2563)
2. **กรมคุมประพฤติ** - ข้อมูลสถิติคดีคุมประพฤติฐานความผิดเมาแล้วขับในช่วงเทศกาลปีใหม่ (ปี 2549 - 2559)

---

## 🛠️ ขั้นตอนที่ 0: นำเข้าไลบรารีและการตั้งค่าเบื้องต้น

In [1]:
import urllib.request
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ssl

# ข้ามการตรวจสอบความปลอดภัย SSL (กรณีเปิดใช้เครื่องโลคอลแล้วพบปัญหาใบรับรองของเว็บราชการ)
ssl._create_default_https_context = ssl._create_unverified_context

print("Libraries imported successfully! Ready to roll.")

Libraries imported successfully! Ready to roll.


## ⛽ ตอนที่ 1: กรมธุรกิจพลังงาน (Department of Energy Business)
### ข้อมูลการใช้น้ำมันเชื้อเพลิงในภาคการขนส่งรายจังหวัด (ปี พ.ศ. 2561 - 2563)
**แหล่งที่มา:** [gdcatalog.energy.go.th](https://gdcatalog.energy.go.th/dataset/73e2f5af-a171-44f9-9d52-4ea755f6b468)

เราจะดึงข้อมูลการใช้น้ำมันเชื้อเพลิงภาคขนส่ง (LPG, ดีเซล, เบนซิน, แก๊สโซฮอล์ 91/95, NGV) รายจังหวัด มาทำการวิเคราะห์แนวโน้มและจัดอันดับจังหวัดที่มีการใช้น้ำมันสูงสุด

In [2]:
fuel_url = "https://gdcatalog.energy.go.th/dataset/73e2f5af-a171-44f9-9d52-4ea755f6b468/resource/ebb99e67-82f2-4dff-b54a-ff7943ef9da0/download/fuel-consumption_tran.csv"

# ดาวน์โหลดข้อมูลโดยเลียนแบบ User-Agent ของบราวเซอร์เพื่อความเสถียร
req = urllib.request.Request(fuel_url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as r:
    df_fuel = pd.read_csv(r)

print(f"โหลดข้อมูลเสร็จสิ้น! ขนาดข้อมูล: {df_fuel.shape[0]} แถว, {df_fuel.shape[1]} คอลัมน์")
df_fuel.head(3)

โหลดข้อมูลเสร็จสิ้น! ขนาดข้อมูล: 231 แถว, 12 คอลัมน์


,Sector,Province,BE_Year,LPG,Low Speed Diesel (LSD),High Speed Diesel (HSD)/Biodiesel,Fuel oil,Gasoline 91,Gasoline 95,Gasohol 91,Gasohol 95,Natural gas
0,Transportation,Bangkok,2561,"287,130,838",0,"5,344,666,516","156,126,840","37,526,000","112,709,456","858,927,410","1,525,303,321","21,235,000,000"
1,Transportation,Samut Prakan,2561,"44,887,031",0,"752,221,018","8,097,217","400,880","7,108,603","107,514,810","142,863,672","6,373,000,000"
2,Transportation,Nonthaburi,2561,"84,196,931",0,"400,565,276",0,0,"10,428,983","92,842,740","145,509,242","4,713,000,000"


🔍 **การทำความสะอาดข้อมูล (Data Cleaning):**
จะเห็นว่า คอลัมน์ตัวเลขมีเครื่องหมายจุลภาค `,` คั่น ทำให้ระบบมองเป็นข้อความ (`object`) เราต้องนำเครื่องหมายจุลภาคออกแล้วแปลงให้เป็นตัวเลขประเภท `float` ก่อนนำไปคำนวณ

In [3]:
# รายชื่อคอลัมน์ที่เป็นค่าปริมาณน้ำมัน
numeric_cols = [
    'LPG', 'Low Speed Diesel (LSD)', 'High Speed Diesel (HSD)/Biodiesel', 
    'Fuel oil', 'Gasoline 91', 'Gasoline 95', 'Gasohol 91', 'Gasohol 95', 'Natural gas'
]

# คลีนคอมมาและแปลงเป็นตัวเลข
for col in numeric_cols:
    df_fuel[col] = df_fuel[col].astype(str).str.replace(',', '', regex=True)
    df_fuel[col] = pd.to_numeric(df_fuel[col], errors='coerce').fillna(0)

# สร้างคอลัมน์กลุ่มประเภทพลังงานหลักเพื่อความเข้าใจง่าย
df_fuel['Diesel'] = df_fuel['Low Speed Diesel (LSD)'] + df_fuel['High Speed Diesel (HSD)/Biodiesel']
df_fuel['Gasoline_Gasohol'] = df_fuel['Gasoline 91'] + df_fuel['Gasoline 95'] + df_fuel['Gasohol 91'] + df_fuel['Gasohol 95']
df_fuel['Gas_LPG_NGV'] = df_fuel['LPG'] + df_fuel['Natural gas']

print("ทำความสะอาดข้อมูลเรียบร้อยแล้ว!")
df_fuel.dtypes

ทำความสะอาดข้อมูลเรียบร้อยแล้ว!


Sector                               object
Province                             object
BE_Year                               int64
LPG                                   int64
Low Speed Diesel (LSD)                int64
High Speed Diesel (HSD)/Biodiesel     int64
Fuel oil                              int64
Gasoline 91                           int64
Gasoline 95                           int64
Gasohol 91                            int64
Gasohol 95                            int64
Natural gas                           int64
Diesel                                int64
Gasoline_Gasohol                      int64
Gas_LPG_NGV                           int64
dtype: object

### 📊 แผนภาพ 1: แนวโน้มการใช้น้ำมันเชื้อเพลิงจำแนกตามประเภทในภาคขนส่งของไทย (ปี 2561 - 2563)
เปรียบเทียบดูว่า การใช้น้ำมันประเภทต่างๆ ได้รับผลกระทบจากสถานการณ์โควิด-19 ในปี 2563 หรือไม่

In [4]:
# รวมข้อมูลยอดขายระดับประเทศแยกตามปี
df_yearly = df_fuel.groupby('BE_Year')[['Diesel', 'Gasoline_Gasohol', 'Gas_LPG_NGV']].sum().reset_index()

# แปลงตารางให้อยู่ในรูป Long format เพื่อนำมาพล็อตด้วย Plotly
df_yearly_melt = df_yearly.melt(
    id_vars=['BE_Year'], 
    value_vars=['Diesel', 'Gasoline_Gasohol', 'Gas_LPG_NGV'],
    var_name='Fuel_Type', 
    value_name='Consumption_Liters'
)

# แปลงประเภทเชื้อเพลิงเป็นภาษาไทยเพื่อให้อ่านง่าย
fuel_thai = {
    'Diesel': 'ดีเซล (Diesel)',
    'Gasoline_Gasohol': 'เบนซิน/แก๊สโซฮอล์ (Gasoline)',
    'Gas_LPG_NGV': 'แก๊ส LPG/NGV'
}
df_yearly_melt['Fuel_Type_TH'] = df_yearly_melt['Fuel_Type'].map(fuel_thai)

# พล็อตกราฟเส้นเปรียบเทียบแนวโน้ม
fig_fuel = px.line(
    df_yearly_melt, 
    x='BE_Year', 
    y='Consumption_Liters', 
    color='Fuel_Type_TH',
    markers=True,
    labels={'BE_Year': 'ปี พ.ศ.', 'Consumption_Liters': 'ปริมาณการใช้ (ลิตร)', 'Fuel_Type_TH': 'ประเภทเชื้อเพลิง'},
    title='📈 แนวโน้มการบริโภคน้ำมันเชื้อเพลิงภาคขนส่งของประเทศไทย (พ.ศ. 2561 - 2563)',
    template='plotly_dark'
)

fig_fuel.update_layout(xaxis=dict(tickmode='array', tickvals=[2561, 2562, 2563]))
fig_fuel.show()

### 📊 แผนภาพ 2: Top 10 จังหวัดที่มีการใช้น้ำมันประเภทเบนซิน/แก๊สโซฮอล์สูงสุด (ปี 2563 - ไม่รวมกรุงเทพมหานคร)
กรุงเทพมหานครจะมีอัตราส่วนที่สูงมากเป็นปกติ เราจึงแยกออกเพื่อดูความต้องการพลังงานในต่างจังหวัดที่มีความสำคัญทางเศรษฐกิจ

In [5]:
# กรองข้อมูลเฉพาะปี 2563 และไม่รวมกรุงเทพฯ
df_prov_2563 = df_fuel[(df_fuel['BE_Year'] == 2563) & (df_fuel['Province'] != 'Bangkok')]

# เลือก 10 อันดับแรก
df_top10 = df_prov_2563.nlargest(10, 'Gasoline_Gasohol')

# พล็อตกราฟแท่งแนวนอน
fig_prov = px.bar(
    df_top10, 
    x='Gasoline_Gasohol', 
    y='Province', 
    orientation='h',
    labels={'Gasoline_Gasohol': 'ปริมาณการใช้เบนซิน/แก๊สโซฮอล์ (ลิตร)', 'Province': 'จังหวัด'},
    title='🏆 Top 10 จังหวัดที่มีปริมาณการใช้เบนซินและแก๊สโซฮอล์สูงสุดในภาคการขนส่ง (ปี พ.ศ. 2563, ไม่รวมกรุงเทพฯ)',
    template='plotly_dark',
    color='Gasoline_Gasohol',
    color_continuous_scale='Viridis'
)

fig_prov.update_layout(yaxis={'categoryorder':'total ascending'})
fig_prov.show()

---

## 👮 ตอนที่ 2: กรมคุมประพฤติ (Department of Probation)
### สถิติการคุมประพฤติคดีเมาแล้วขับ ช่วงเทศกาลปีใหม่ (ปี พ.ศ. 2549 - 2559)
**แหล่งที่มา:** [data.go.th](https://data.go.th/dataset/d2b03f35-779c-4a11-8f3a-910b07669595)

เราจะดึงข้อมูลสถิติคดีเมาแล้วขับในช่วงเทศกาลปีใหม่ที่ถูกส่งตัวมาคุมประพฤติเพื่อนำมาวิเคราะห์แนวโน้มระดับประเทศในรอบ 10 ปี

In [6]:
probation_url = "https://data.go.th/dataset/d2b03f35-779c-4a11-8f3a-910b07669595/resource/9a3772a0-a4cd-4e54-aa77-98e6edc9f080/download/_.._2549_-_._2559.csv"

req = urllib.request.Request(probation_url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as r:
    df_prob = pd.read_csv(r, encoding='tis-620')

print("โหลดข้อมูลสถิติคดีคุมประพฤติเสร็จสิ้น!")
df_prob

โหลดข้อมูลสถิติคดีคุมประพฤติเสร็จสิ้น!


,สถิติการคุมความประพฤติคดีเมาแล้วขับ ช่วงปีใหม่,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ปี,ช่วงเวลา,จำนวนผู้ถูกคุมฯ,เพิ่มขึ้น/ลดลง,คิดเป็น,จังหวัดที่สูงที่สุด,NaN,NaN,NaN,NaN
2,2549,-,3.410 คดี,NaN,NaN,,NaN,NaN,NaN,NaN
3,2550,28 ธ.ค.  3 ม.ค. 50,"4,774 คดี","เพิ่มขึ้น 1,364 คดี",คิดเป็น 40 %,"จังหวัดที่มีคดีมากได้แก่ กทม. 1,306 คดี บุรีร...",NaN,NaN,NaN,NaN
4,2551,28 ธ.ค. - 3 ม.ค. 51,"3,047 คดี","ลดลง 1,727 คดี",คิดเป็น 36.17 %,จังหวัดที่มีคดีมากได้แก่ กทม. 348 คดี นครราชสี...,NaN,NaN,NaN,NaN
5,2552,ไม่ได้มีการเก็บ,-,NaN,NaN,-,NaN,NaN,NaN,NaN
6,2553,29 ธ.ค.52 - 4 ม.ค. 53,"3,802 คดี",เพิ่มขึ้น 755 คดี,คิดเป็น 24.77 %,จังหวัดที่มีคดีมากได้แก่ กทม. 512 คดี สกลนคร 2...,NaN,NaN,NaN,NaN
7,2554,29 ธ.ค.53  4 ม.ค.54,"3,379 คดี",ลดลง 423 คดี,คิดเป็น 11.12 %,จังหวัดที่มีคดีมาก ได้แก่ สุรินทร์ 353 คดี นคร...,NaN,NaN,NaN,NaN
8,2555,29 ธ.ค.54  4 ม.ค.55,"3,768 คดี",เพิ่มขึ้น 385 คดี,คิดเป็น 11.51 %,จังหวัดที่มีคดีมาก ได้แก่ สุรินทร์ 261 คดี ชลบ...,NaN,NaN,NaN,NaN
9,2556,27 ธ.ค.55  2 ม.ค.56,"4,635 คดี",เพิ่มขึ้น 867 คดี,คิดเป็น 23 %,จังหวัดที่มีคดีมาก ได้แก่ กทม. 398 คดี มหาสารค...,NaN,NaN,NaN,NaN


🔍 **การทำความสะอาดข้อมูล (Data Cleaning):**
ข้อมูลไฟล์ CSV นี้มาจากตารางรายงานรูปแบบเก่า ทำให้อ่านชื่อคอลัมน์เพี้ยน และมีแถวว่างหรือแถวที่ไม่เกี่ยวข้องอยู่ด้วย:
1. เลือกเฉพาะแถวที่มีตัวเลขปี พ.ศ. ชัดเจน
2. เปลี่ยนชื่อคอลัมน์ให้สื่อความหมายง่ายขึ้น
3. ล้างสัญลักษณ์ที่ปนเปื้อนในช่องตัวเลข (เช่น ลบคำว่า `คดี` และเครื่องหมายคอมมา `,`) แล้วแปลงค่าเป็น `integer`

In [7]:
# คัดเลือกคอลัมน์ที่ต้องการ และคัดแถวที่มีข้อมูลปี พ.ศ. (คัดแถวที่เป็นตัวเลขช่วง 2549 - 2559)
df_clean_prob = df_prob.copy()
df_clean_prob.columns = ['Year_TH', 'Period', 'Cases', 'Change', 'Percent', 'Top_Province', 'Un1', 'Un2', 'Un3', 'Un4']
df_clean_prob = df_clean_prob[['Year_TH', 'Period', 'Cases', 'Top_Province']]

# เอาแถวที่เป็นค่าว่างหรือตัวหนังสือสรุปหัวเรื่องออก
df_clean_prob = df_clean_prob[df_clean_prob['Year_TH'].astype(str).str.isdigit()]
df_clean_prob['Year_TH'] = df_clean_prob['Year_TH'].astype(int)

# ทำความสะอาดข้อมูลคอลัมน์ 'Cases'
# เอาช่องว่าง, คอมมา, และคำว่า 'คดี' ออก
df_clean_prob['Cases'] = df_clean_prob['Cases'].astype(str).str.replace(',', '').str.replace('คดี', '').str.strip()
df_clean_prob['Cases'] = pd.to_numeric(df_clean_prob['Cases'], errors='coerce')

# จัดการกับค่าสูญหายหรือข้อมูลที่ไม่ได้เก็บ (เช่น ปี 2552 ระบุว่า 'ไม่ได้มีการเก็บ') ให้ใส่เป็นค่าเฉลี่ยหรือ 0
# ในที่นี้แปลงค่า NaN เป็น 0
df_clean_prob['Cases'] = df_clean_prob['Cases'].fillna(0)

print("ข้อมูลหลังจากจัดโครงสร้างเสร็จสิ้น:")
df_clean_prob

ข้อมูลหลังจากจัดโครงสร้างเสร็จสิ้น:


,Year_TH,Period,Cases,Top_Province
2,2549,-,3.41,
3,2550,28 ธ.ค.  3 ม.ค. 50,4774.00,"จังหวัดที่มีคดีมากได้แก่ กทม. 1,306 คดี บุรีร..."
4,2551,28 ธ.ค. - 3 ม.ค. 51,3047.00,จังหวัดที่มีคดีมากได้แก่ กทม. 348 คดี นครราชสี...
5,2552,ไม่ได้มีการเก็บ,0.00,-
6,2553,29 ธ.ค.52 - 4 ม.ค. 53,3802.00,จังหวัดที่มีคดีมากได้แก่ กทม. 512 คดี สกลนคร 2...
7,2554,29 ธ.ค.53  4 ม.ค.54,3379.00,จังหวัดที่มีคดีมาก ได้แก่ สุรินทร์ 353 คดี นคร...
8,2555,29 ธ.ค.54  4 ม.ค.55,3768.00,จังหวัดที่มีคดีมาก ได้แก่ สุรินทร์ 261 คดี ชลบ...
9,2556,27 ธ.ค.55  2 ม.ค.56,4635.00,จังหวัดที่มีคดีมาก ได้แก่ กทม. 398 คดี มหาสารค...
10,2557,27 ธ.ค.56  2 ม.ค.57,3797.00,จังหวัดที่มีคดีมาก ได้แก่ สกลนคร 407 คดี ชลบุร...
11,2558,30 ธ.ค.57 -5 ม.ค.58,4059.00,จังหวัดที่มีคดีมาก ได้แก่ ชลบุรี 333 คดี มหาสา...


### 📊 แผนภาพ 3: แนวโน้มจำนวนคดีคุมประพฤติเมาแล้วขับ ช่วงเทศกาลปีใหม่ (พ.ศ. 2549 - 2559)
แสดงแนวโน้มสถิติคดีเมาขับที่ถูกคุมประพฤติทั่วประเทศในรอบทศวรรษ

In [8]:
# พล็อตกราฟแท่งเพื่อดูจำนวนคดีรายปี พร้อมระบุกราฟแนวโน้ม
# กรองแถวที่ปี 2552 ซึ่งเป็น 0 ออก เพื่อไม่ให้กราฟวูบโดยไม่จำเป็น หรือตั้งกำกับในคำอธิบาย
df_plot_prob = df_clean_prob[df_clean_prob['Cases'] > 0]

fig_prob = px.bar(
    df_plot_prob,
    x='Year_TH',
    y='Cases',
    labels={'Year_TH': 'ปี พ.ศ.', 'Cases': 'จำนวนคดีสะสม (คดี)'},
    title='🚨 สถิติคดีผู้ถูกคุมความประพฤติเมาแล้วขับช่วงเทศกาลปีใหม่ (พ.ศ. 2549 - 2559)',
    text='Cases',
    template='plotly_dark',
    color='Cases',
    color_continuous_scale='Reds'
)

fig_prob.update_traces(textposition='outside')
fig_prob.update_layout(xaxis=dict(tickmode='linear', dtick=1))
fig_prob.show()

---

## 💡 🧠 ไอเดียไอเดียสำหรับการจัดทำผลงานจริง (Hackathon Idea Sparks)

หากคุณต้องใช้ข้อมูลจาก 2 หน่วยงานนี้ หรือนำไปต่อยอดกับข้อมูลที่คุณถืออยู่ ลองจับคู่ประเด็นที่ท้าทายเหล่านี้:

1. **⚡ การทำนายความคุ้มค่าของการลงทุนจุดชาร์จ EV (EV Charging Station Predictor)**
   - **วิกฤต:** ปริมาณการจำหน่ายน้ำมันเชื้อเพลิงรายจังหวัดบอกปริมาณการใช้งานในพื้นที่ หากนำมาคำนวณร่วมกับสถิติรถยนต์จดทะเบียนไฟฟ้า (EV) จากกรมการขนส่งทางบก จะระบุได้ว่าจังหวัดใดที่มีการใช้น้ำมันแก๊สโซฮอล์/เบนซินลดลงรวดเร็วที่สุด และจุดใดที่เหมาะสมกับการสร้างสถานีชาร์จเพื่อดักตลาดอนาคต

2. **🚨 ระบบประเมินความเสี่ยงอุบัติเหตุช่วงเทศกาล (Dynamic Festival Risk Zone Dashboard)**
   - **วิกฤต:** นำสถิติคดีเมาขับรายปีของกรมคุมประพฤติ มารวมกับข้อมูลการจราจรหนาแน่นในระบบของกรมทางหลวง เพื่อประเมินความเสี่ยงล่วงหน้าแบบเกือบเรียลไทม์ และส่งการเตือนภัยให้กับประชาชนหรือด่านตรวจท้องถิ่น

3. **💰 สถิติร้องเรียนสู่การปรับปรุงความเชื่อมั่นในสถานบริการพลังงาน (DOEB Clean Fuel & Safety Index)**
   - **วิกฤต:** ใช้ข้อมูลเรื่องร้องเรียนสถานประกอบการพลังงาน (LPG/ปั๊มน้ำมัน) ของกรมธุรกิจพลังงาน ร่วมกับความหนาแน่นเชิงพื้นที่ เพื่อสร้างดัชนีคะแนนความปลอดภัยในปั๊มน้ำมันแก๊สทั่วประเทศ